# PravahX — Delft3D FM Smoke Package Generator

Generates `pravahx_smoke_linux_package.tar.gz` from the Deltares c019 test case to test the runner independently of the model generator.


In [ ]:
%%python
import os, tarfile, shutil

src_dir = '/content/delft3d_src/test/tests/FMtests/f05_boundary_conditions/c019_waterlevel_bc_cmp_varying'
dest_dir = '/content/pravahx_smoke_case'

if not os.path.exists(src_dir):
    print(f"\u2717 Source directory {src_dir} not found. Please run this in the builder session.")
    exit(1)

if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
shutil.copytree(src_dir, dest_dir)

# Apply MDU edits
mdu_path = os.path.join(dest_dir, 'c019_waterlevel_bc_cmp_varying.mdu')
with open(mdu_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

obsolete_keys = ['TransportMethod', 'Qhrelax', 'Jaorgsethu', 'EffectSpiral', 'Gapres', 'WaveNikuradse', 'Writebalancefile']
new_lines = []
for line in lines:
    lower_line = line.lower().strip()
    is_obs = any(lower_line.startswith(k.lower() + '=') or lower_line.startswith(k.lower() + ' ') for k in obsolete_keys)
    if is_obs:
        new_lines.append(f"# {line}")
        continue
    if lower_line.startswith('obsfile'):
        parts = line.split('=', 1)
        if len(parts) > 1:
            new_lines.append(f"{parts[0]}= \n")
            continue
    if lower_line.startswith('mapformat'):
        new_lines.append("MapFormat = 4\n")
        continue
    if lower_line.startswith('mapinterval'):
        new_lines.append("MapInterval = 600\n")
        continue
    new_lines.append(line)

with open(mdu_path, 'w', encoding='utf-8') as f:
    f.writelines(new_lines)

print("\u2713 Case copied and MDU patched.")


In [ ]:
%%bash
cat << 'EOF' > /content/pravahx_smoke_case/run_linux.sh
#!/bin/bash
set -e
echo "=== Running Delft3D FM Simulation ==="

# Build LD_LIBRARY_PATH dynamically
INTEL_LIBS_DIR=""
if [ -d "/opt/intel" ]; then
    echo "Finding Intel libraries..."
    DIRS=$(find /opt/intel -type f \( -name "libifcore.so*" -o -name "libimf.so*" -o -name "libiomp5.so*" -o -name "libmpi.so.12*" \) -exec dirname {} \; | sort -u | tr '\n' ':' | sed 's/:$//')
    if [ -n "$DIRS" ]; then
        INTEL_LIBS_DIR=":$DIRS"
    fi
fi

export LD_LIBRARY_PATH=/content/delft3d_bin/lib${INTEL_LIBS_DIR}:$LD_LIBRARY_PATH
echo "Final LD_LIBRARY_PATH=$LD_LIBRARY_PATH"

echo "Checking missing dependencies with ldd:"
ldd /content/delft3d_bin/bin/dflowfm | grep "not found" || echo "All OK."

# Run D-Flow FM
/content/delft3d_bin/bin/dflowfm --autostartstop c019_waterlevel_bc_cmp_varying.mdu || exit 1

# Generate Hash Manifest of output NetCDF and log files
echo "=== Generating Hash Manifest ==="
python3 -c "
import hashlib, json, os, glob
manifest = {'files': {}}
output_dir = 'DFM_OUTPUT_c019_waterlevel_bc_cmp_varying'
if os.path.exists(output_dir):
    for root, _, files in os.walk(output_dir):
        for f in files:
            p = os.path.join(root, f)
            with open(p, 'rb') as fp:
                manifest['files'][p] = hashlib.sha256(fp.read()).hexdigest()
with open(os.path.join(output_dir, 'manifest.json'), 'w') as mf:
    json.dump(manifest, mf, indent=2)
"
echo "=== Simulation Complete. Manifest Generated. ==="
EOF
chmod +x /content/pravahx_smoke_case/run_linux.sh
echo "\u2713 run_linux.sh created."

cd /content
tar -czvf pravahx_smoke_linux_package.tar.gz -C /content/pravahx_smoke_case .
cp pravahx_smoke_linux_package.tar.gz /content/drive/MyDrive/PravahX/
echo "\u2713 Package created and copied to Drive."
